In [1]:
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10
import torchvision
import numpy as np
import random

import torch
import torch.nn.functional as F
import cl_gym as cl

import sys
import os

init_path = os.path.abspath('.')
new_path = init_path
while True:
    if new_path[-3:] == "FSW":
        sys.path.append(new_path)
        break
    new_path = os.path.abspath('..')
    os.chdir(new_path)


seed = 0

np.random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False
torch.set_num_threads(8)

def make_params() -> dict:
    import os
    from pathlib import Path
    import uuid

    params = {
            # dataset
            'dataset': "Bios",
            'fairness_agg': 'mean',
            # 'model': 'MLP',

            # benchmark
            'seed': seed,
            'num_tasks': 5,
            'epochs_per_task': 10,
            'per_task_examples': np.inf,
            # 'per_task_examples': 10000,
            'per_task_memory_examples': 320,
            'batch_size_train': 64,
            'batch_size_memory': 64,
            'batch_size_validation': 256,
            'tau': 5.0,

            # algorithm
            'optimizer': 'sgd',
            'learning_rate': 2e-5,
            'momentum': 0.9,
            'learning_rate_decay': 1.0,
            'criterion': torch.nn.CrossEntropyLoss(),
            # 'criterion': torch.nn.BCEWithLogitsLoss(),

            'device': torch.device('cuda:5' if torch.cuda.is_available() else 'cpu'),
             
            # sample selection
            'alpha': 0.0005,
            'metric' : "EO",
            'lambda': 1.0,
            'lambda_old': 0.0,

            # postprocessing
            # "post_processing": "eps_fairness"

              }
    

#     trial_id = str(uuid.uuid4())
    trial_id = f"demo/dataset={params['dataset']}/seed={params['seed']}_epoch={params['epochs_per_task']}_lr={params['learning_rate']}_tau={params['tau']}_alpha={params['alpha']}"
    if params['lambda'] != 0:
        trial_id+=f"_lmbd_{params['lambda']}_lmbdold_{params['lambda_old']}"
    params['trial_id'] = trial_id
    params['output_dir'] = os.path.join("./outputs/{}".format(trial_id))
    print(f"output_dir={params['output_dir']}")
    Path(params['output_dir']).mkdir(parents=True, exist_ok=True)

    return params

params = make_params()

output_dir=./outputs/demo/dataset=Bios/seed=0_epoch=10_lr=2e-05_tau=5.0_alpha=0.0005_lmbd_1.0_lmbdold_0.0


In [2]:
"MNIST" in params['dataset']

False

In [3]:
from datasets.bios import Bios

if params['dataset'] in ["Bios"]:
    benchmark = Bios(num_tasks=params['num_tasks'],
                        per_task_memory_examples=params['per_task_memory_examples'],
                        per_task_examples = params['per_task_examples'],
                        max_length = 128,
                        random_class_idx = False)
    input_dim = (12)
    class_idx = benchmark.class_idx
    num_classes = len(class_idx)

In [4]:
from backbones.bert import BertClassifier

backbone = BertClassifier(num_classes, benchmark.bert_config['model'], params)

from trainers import FairContinualTrainer
from trainers.fair_trainer import FairContinualTrainer2
from metrics import FairMetricCollector
from metrics import MetricCollector2

from algorithms import Heuristic3

algorithm = Heuristic3(backbone, benchmark, params, requires_memory=True)

metric_manager_callback = FairMetricCollector(num_tasks=params['num_tasks'],
                                                        eval_interval='epoch',
                                                        epochs_per_task=params['epochs_per_task'])
# metric_manager_callback = MetricCollector2(num_tasks=params['num_tasks'],
#                                                         eval_interval='epoch',
#                                                         epochs_per_task=params['epochs_per_task'])
# from trainers.baselines import BaseMemoryContinualTrainer as ContinualTrainer
from trainers.baselines import BaseContinualTrainer as ContinualTrainer

# trainer = ContinualTrainer(algorithm, params, callbacks=[metric_manager_callback]) # type: ignore
# 
trainer = FairContinualTrainer2(algorithm, params, callbacks=[metric_manager_callback])


In [5]:
if params['fairness_agg'] == "mean":
    agg = np.mean
elif params['fairness_agg'] == "max":
    agg = np.max
else:
    raise NotImplementedError

fairness_metrics = ["std", "EER", "EO", "DP"]
for metric in metric_manager_callback.meters:
    if metric in fairness_metrics:
        metric_manager_callback.meters[metric].agg = agg


In [6]:
trainer.run()
print("final avg-acc", metric_manager_callback.meters['accuracy'].compute_final())
print("final avg-forget", metric_manager_callback.meters['forgetting'].compute_final())

---------------------------- Task 1 -----------------------
[1] Eval metrics for task 1 >> {'accuracy': 0.6976970955472479, 'loss': 0.0023360830628216733, 'std': 0.3486153692888529, 'EER': -1, 'EO': [0.009858539611530048, 0.0032041488897477377, 0.006431999037552901, 0.19921163396284214, 0.033704788888395676], 'DP': -1, 'accuracy_s0': 0.6709935706348011, 'accuracy_s1': 0.7027688017569454, 'classwise_accuracy': {3: array([5016, 6081]), 0: array([28246, 29527]), 4: array([  54, 4987]), 1: array([8764, 9641]), 2: array([6416, 8151])}, 'DP_ingredients': {'class_pred_count_s0': {3: 4271, 0: 19352, 1: 2650, 2: 5177, 4: 26, 18: 1}, 'class_pred_count_s1': {0: 14076, 1: 7430, 2: 2861, 3: 2506, 4: 37}, 'class_pred_count': {3: 6777, 0: 33428, 1: 10080, 2: 8038, 4: 63, 18: 1}, 'count_s0': 31477, 'count_s1': 26910, 'count': 58387}}
[2] Eval metrics for task 1 >> {'accuracy': 0.885030938224056, 'loss': 0.0011677482028426222, 'std': 0.05321173088812073, 'EER': -1, 'EO': [0.006828657962005824, 0.027901

In [7]:
import copy
task_weight = copy.deepcopy(algorithm.weight_all)

num_bin = 20
np.arange(0+1/num_bin, 1+1/num_bin, 1/num_bin)

def bin(w: np.array, num_bin=20):
    out = dict()
    for r in np.arange(0+1/num_bin, 1+1/num_bin, 1/num_bin):
        r = np.round(r, 2)
        out[r] = np.sum(np.logical_and(w<=r, r-1/num_bin<w))
    out[1/num_bin] += np.sum(w==0)
    kk = list(out.keys())
    for k in kk:
        if out[k] == 0:
            del(out[k])
    return out


binned_weight = dict()
for i, wt in enumerate(task_weight):
    if i==0:
        continue
    print(f"task:{i+1}")
    binned_weight[i+1] = list()
    for we in wt:
        binned_weight[i+1].append({k: bin(we[k]) for k in we})




task:2
task:3
task:4
task:5


In [9]:
binned_weight

{2: [{0: {0.05: 32, 1.0: 23125}, 1: {0.05: 373, 0.15: 1, 1.0: 29256}},
  {0: {0.05: 2886, 0.3: 1, 0.75: 1, 1.0: 20269}, 1: {0.05: 3996, 1.0: 25634}},
  {0: {0.05: 3929, 0.7: 1, 1.0: 19227}, 1: {0.05: 5197, 0.3: 1, 1.0: 24432}},
  {0: {0.05: 3702, 0.6: 1, 0.8: 1, 1.0: 19453},
   1: {0.05: 3762, 0.9: 1, 0.95: 1, 1.0: 25866}},
  {0: {0.05: 3980, 0.55: 1, 0.8: 1, 1.0: 19175},
   1: {0.05: 4698, 0.35: 1, 0.65: 1, 1.0: 24930}},
  {0: {0.05: 3304, 0.15: 1, 0.3: 1, 0.85: 1, 1.0: 19850},
   1: {0.05: 4049, 0.7: 1, 1.0: 25580}},
  {0: {0.05: 4513, 0.2: 1, 0.4: 1, 0.55: 1, 1.0: 18641},
   1: {0.05: 4589, 1.0: 25041}},
  {0: {0.05: 7914, 1.0: 15243},
   1: {0.05: 14575, 0.25: 1, 0.3: 1, 0.65: 1, 1.0: 15052}},
  {0: {0.05: 12341, 0.1: 1, 0.9: 1, 1.0: 10814},
   1: {0.05: 11969, 0.6: 1, 1.0: 17660}},
  {0: {0.05: 8329, 0.55: 2, 1.0: 14826},
   1: {0.05: 15286, 0.35: 1, 0.75: 1, 0.8: 1, 1.0: 14341}}],
 3: [{0: {0.05: 86, 1.0: 13588}, 1: {0.05: 42, 1.0: 11873}},
  {0: {0.05: 53, 1.0: 13621}, 1: {0.05:

In [10]:
metric_manager_callback.meters['accuracy'].get_data()

array([[0.917, 0.   , 0.   , 0.   , 0.   ],
       [0.742, 0.862, 0.   , 0.   , 0.   ],
       [0.707, 0.795, 0.876, 0.   , 0.   ],
       [0.722, 0.754, 0.748, 0.9  , 0.   ],
       [0.697, 0.72 , 0.742, 0.881, 0.679]])

In [11]:
np.mean(metric_manager_callback.meters['accuracy'].compute_overall())

0.807298525883095

In [12]:
[np.round(x, 3) for x in metric_manager_callback.meters['EO'].compute_overall()]

[0.035, 0.084, 0.072, 0.073, 0.082]

In [13]:
np.mean(metric_manager_callback.meters['EO'].compute_overall())

0.06902487203458807

In [14]:
[np.round(x, 3) for x in metric_manager_callback.meters['DP'].compute_overall()]

[0.037, 0.027, 0.019, 0.015, 0.011]

In [15]:
np.mean(metric_manager_callback.meters['DP'].compute_overall())

0.021732696498277367